# 🎮 Face Recognition Game: Pro Edition
**ระบบเกมจัดเต็มตามคำขอ:**
1. ใส่เพลงประกอบ (MP3)
2. เลือกระดับความยาก (ตุ่ยดุ้ย, ซีเล็ง, สลิ้งแตก)
3. เลือกผู้เล่น (Target) จากดาต้าเซตได้
4. กำหนดเวลาเล่นเกมรวมได้ (เช่น เล่นรอบละ 60 วินาที)
5. นับแต้มแข่งกัน!

⚠️ **ย้ำอีกครั้ง:** โค้ดนี้ต้องนำไปรันบนคอมพิวเตอร์ของคุณเองนะครับ (รันบน VS Code หรือ Jupyter Local)


In [ ]:
# 1. ติดตั้งไลบรารีที่จำเป็น
# !pip install facenet-pytorch scikit-learn joblib opencv-python Pillow numpy pygame


In [ ]:
# 2. นำเข้าไลบรารีและโหลดโมเดล
import cv2
import torch
import numpy as np
import joblib
import time
import random
import os
import pygame  # สำหรับเล่นเพลง
from facenet_pytorch import MTCNN, InceptionResnetV1
from PIL import Image

# ฟังก์ชันเล่นเพลง
def play_bgm(music_file="bg_game.mp3"):
    try:
        if os.path.exists(music_file):
            pygame.mixer.init()
            pygame.mixer.music.load(music_file)
            pygame.mixer.music.set_volume(0.5)
            pygame.mixer.music.play(-1) # เล่นวนซ้ำ
            print(f"🎵 กำลังเล่นเพลง: {music_file}")
        else:
            print(f"⚠️ ไม่พบไฟล์เพลง {music_file} (ระบบจะรันเกมแบบไม่มีเสียง)")
    except Exception as e:
        print(f"⚠️ ไม่สามารถเล่นเพลงได้: {e}")

def stop_bgm():
    try:
        pygame.mixer.music.stop()
    except:
        pass

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'💻 ใช้หน่วยประมวลผล: {device}')

print('กำลังโหลดโมเดล FaceNet...')
mtcnn = MTCNN(image_size=160, margin=20, keep_all=True, select_largest=False, post_process=True, device=device)
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

clf_path = 'facenet_svm_model.pkl'
le_path = 'label_encoder.pkl'

if os.path.exists(clf_path) and os.path.exists(le_path):
    clf = joblib.load(clf_path)
    le = joblib.load(le_path)
    all_names = list(le.classes_)
    print(f'✅ โหลดโมเดลสำเร็จ! รายชื่อในระบบ: {all_names}')
else:
    raise FileNotFoundError('❌ ไม่พบไฟล์โมเดล!')


In [ ]:
# 3. เมนูตั้งค่าก่อนเริ่มเกม
print("="*40)
print("     ⚙️ เมนูตั้งค่าเกม ⚙️")
print("="*40)

# 1. เลือกผู้เล่นเป้าหมาย
print(f"รายชื่อทั้งหมดที่มี: {all_names}")
selected_names_input = input("พิมพ์ชื่อคนที่ต้องการใช้เล่น (คั่นด้วยลูกน้ำ) หรือ [กด Enter] เพื่อใช้ทุกคน: ")
if selected_names_input.strip() == "":
    active_names = all_names
else:
    active_names = [n.strip() for n in selected_names_input.split(',')]
    active_names = [n for n in active_names if n in all_names]
    if len(active_names) == 0:
        active_names = all_names

# 2. เลือกระดับความยาก (กำหนดความแม่นยำขั้นต่ำ)
print("\nเลือกระดับความยาก:")
print("1. ตุ่ยดุ้ย (ง่าย - ความแม่นยำ 80%)")
print("2. ซีเล็ง (ปานกลาง - ความแม่นยำ 90%)")
print("3. สลิ้งแตก (ยาก - ความแม่นยำ 95%)")
diff_input = input("เลือก [1/2/3] (ค่าเริ่มต้น 2): ")
if diff_input == '1': confidence_needed = 0.80
elif diff_input == '3': confidence_needed = 0.95
else: confidence_needed = 0.90

# 3. กำหนดเวลารวมของเกม
time_input = input("\nกำหนดเวลาเล่นรวมของตานี้ (วินาที) เช่น 60 [ค่าเริ่มต้น 60]: ")
try:
    total_game_time = float(time_input)
except:
    total_game_time = 60.0

# 4. ใส่ชื่อไฟล์เพลง
music_input = input("\nพิมพ์ชื่อไฟล์เพลง หรือ [กด Enter] เพื่อใช้ค่าเริ่มต้น (bg_game.mp3): ")
if music_input.strip() == "":
    music_input = "bg_game.mp3"

print("\n✅ ตั้งค่าเสร็จสิ้น! เตรียมเปิดกล้อง...")


In [ ]:
# 4. ระบบเกมหลัก
def play_face_game():
    if music_input.strip() != "":
        play_bgm(music_input.strip())
        
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("❌ ไม่สามารถเปิดกล้องได้")
        return

    score = 0
    target_name = random.choice(active_names)
    start_time = time.time()
    game_over = False

    print("🎮 เริ่มเกม! ทำแต้มให้ได้มากที่สุด! กด [Q] เพื่อออก")

    while True:
        ret, frame = cap.read()
        if not ret: break
        
        frame = cv2.flip(frame, 1)
        display_frame = frame.copy()
        
        elapsed = time.time() - start_time
        time_left = max(0, total_game_time - elapsed)
        
        if time_left <= 0:
            game_over = True

        if game_over:
            cv2.putText(display_frame, "TIME'S UP!", (120, 200), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 255), 5)
            cv2.putText(display_frame, f"Final Score: {score}", (150, 280), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 255), 3)
            cv2.putText(display_frame, "Press 'Q' to Quit", (180, 350), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
            cv2.imshow("Face Game", display_frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
            continue

        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(img_rgb)
        
        boxes, probs = mtcnn.detect(pil_img)
        
        highest_conf_for_target = 0.0

        if boxes is not None:
            faces = mtcnn(pil_img)
            if faces is not None:
                faces_tensor = faces.to(device)
                if faces_tensor.dim() == 3:
                    faces_tensor = faces_tensor.unsqueeze(0)
                    
                with torch.no_grad():
                    embeddings = resnet(faces_tensor).cpu().numpy()
                
                preds = clf.predict(embeddings)
                pred_probs = clf.predict_proba(embeddings)
                
                for i, box in enumerate(boxes):
                    if i >= len(preds) or probs[i] < 0.90: continue
                    w, h = box[2] - box[0], box[3] - box[1]
                    if w < 40 or h < 40: continue
                    
                    prob_max = np.max(pred_probs[i])
                    name = le.inverse_transform([preds[i]])[0]
                    
                    if name == target_name:
                        if prob_max > highest_conf_for_target:
                            highest_conf_for_target = prob_max
                    
                    color = (0, 255, 255) if name == target_name else (200, 200, 200)
                    x1, y1, x2, y2 = map(int, box)
                    cv2.rectangle(display_frame, (x1, y1), (x2, y2), color, 2)
                    cv2.putText(display_frame, f"{name} {prob_max*100:.0f}%", (x1, max(0, y1-10)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

        # ผ่านด่าน!
        if highest_conf_for_target >= confidence_needed:
            score += 1
            cv2.rectangle(display_frame, (0,0), (display_frame.shape[1], display_frame.shape[0]), (0,255,0), 15)
            cv2.putText(display_frame, "+1 POINT!", (180, 250), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 0), 5)
            cv2.imshow("Face Game", display_frame)
            cv2.waitKey(500)
            target_name = random.choice(active_names)

        # UI
        cv2.putText(display_frame, f"TARGET: {target_name}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 3)
        cv2.putText(display_frame, f"Time: {time_left:.1f}s", (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255) if time_left < 10 else (255, 255, 255), 2)
        cv2.putText(display_frame, f"Score: {score}", (20, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)
        
        # Progress bar
        bar_x, bar_y, bar_w, bar_h = 20, 430, 400, 30
        cv2.rectangle(display_frame, (bar_x, bar_y), (bar_x + bar_w, bar_y + bar_h), (50, 50, 50), -1)
        fill_w = int(bar_w * highest_conf_for_target)
        bar_color = (0, 255, 0) if highest_conf_for_target >= confidence_needed else (0, 165, 255)
        cv2.rectangle(display_frame, (bar_x, bar_y), (bar_x + fill_w, bar_y + bar_h), bar_color, -1)
        cv2.rectangle(display_frame, (bar_x, bar_y), (bar_x + bar_w, bar_y + bar_h), (255, 255, 255), 2)
        
        line_goal_x = bar_x + int(bar_w * confidence_needed)
        cv2.line(display_frame, (line_goal_x, bar_y - 5), (line_goal_x, bar_y + bar_h + 5), (0, 0, 255), 2)
        cv2.putText(display_frame, f"Goal ({int(confidence_needed*100)}%)", (line_goal_x - 30, bar_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

        cv2.imshow("Face Game", display_frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    stop_bgm()

play_face_game()
